# Match injections to recovered sources

Uses the outputs of the `copu-cat-*` commands in `DATA_DIR` (converted inputs, `hdrs.feather`, and optionally `posterior_snr.csv`):
matches injections to posteriors, makes the one-to-one assignment, and plots match status, purity and completeness.
Results are saved in `DATA_DIR` for `ambiguous_matches.ipynb`.

In [ ]:
# Where the data live (see copu_cat.config for the layout) and where plots are saved.
DATA_DIR = '../data'
PLOTS_DIR = '../plots'

In [ ]:
import glob
from copu_cat.galactic_binary import GalacticBinary
from copu_cat.match_injections import get_all_matches
from copu_cat.utils import get_galactic_binary_names
import pandas as pd
import numpy as np
import os
from copu_cat.assignment import build_match_table, assign_one_to_one, injection_summary, posterior_summary
from copu_cat.catalog_metrics import posterior_medians, purity, completeness
from copu_cat.plotting import plot_match_hist2d, plot_fraction_map

In [ ]:
hdrs = pd.read_feather(f'{DATA_DIR}/hdrs.feather')

In [ ]:
# Preview mode: only check posteriors that already have HDRs (run_hdrs.py may still be running).
# For the final result, set PREVIEW = False once all HDRs are computed and combined.
PREVIEW = False
names = sorted(hdrs['Name'].unique()) if PREVIEW else get_galactic_binary_names(data_dir=DATA_DIR)
matches = get_all_matches(hdrs, threshold=0.99, names=names, data_dir=DATA_DIR)

In [ ]:
# count the number of posteriors that have at least one matching injection
num_matches = sum(len(m) >= 1 for m in matches.values())
print(f'Number of posteriors with at least one matching injection: {num_matches} out of {len(names)}'
      + (' (preview: posteriors with HDRs so far)' if PREVIEW else ''))

## Settings for matching and one-to-one assignment
- `CRITERION`: `'6d'` = injection inside the `THRESHOLD` credible region of the 6D HDR; `'1d'` = all six 1D HDRs below the Bonferroni-corrected threshold.
- `ASSIGNMENT`: `'max_snr'` = highest catalog SNR first, each takes its highest-SNR available injection; `'min_hdr'` = largest one-to-one set with the lowest total 6D HDR.
- `'max_snr'` needs the catalog SNR of each posterior, read from `POSTERIOR_SNR_FILE` (csv with columns `Name`, `SNR`).

In [ ]:
THRESHOLD = 0.99
CRITERION = '6d'          # '6d' (paper) or '1d'
ASSIGNMENT = 'max_snr'    # 'max_snr' (paper, needs POSTERIOR_SNR_FILE) or 'min_hdr'
POSTERIOR_SNR_FILE = f'{DATA_DIR}/posterior_snr.csv'
CATALOG_SNR = 'SNR_median'  # 'SNR_median' = median SNR over posterior draws (paper); 'SNR' = SNR at the parameter-wise median
tag = f'{CRITERION}_{ASSIGNMENT}' + ('_preview' if PREVIEW else '')

# median recovered parameters of every posterior (computed once, then cached)
posteriors = posterior_medians(cache=f'{DATA_DIR}/posterior_medians.feather', data_dir=DATA_DIR)
if os.path.exists(POSTERIOR_SNR_FILE):
    snr_table = pd.read_csv(POSTERIOR_SNR_FILE, dtype={'Name': str})
    if CATALOG_SNR not in snr_table:
        raise KeyError(f"{CATALOG_SNR} not in {POSTERIOR_SNR_FILE} (columns: {list(snr_table.columns)}); "
                       "SNR_median needs compute_snr.py posteriors --n-samples N")
    # the catalog SNR used everywhere below (greedy ordering and purity binning) is the 'SNR' column
    posteriors = posteriors.merge(snr_table[['Name', CATALOG_SNR]].rename(columns={CATALOG_SNR: 'SNR'}),
                                  on='Name', how='left')
HAVE_POSTERIOR_SNR = 'SNR' in posteriors
if ASSIGNMENT == 'max_snr' and not HAVE_POSTERIOR_SNR:
    raise FileNotFoundError(f"ASSIGNMENT='max_snr' needs catalog SNRs in {POSTERIOR_SNR_FILE}")
if PREVIEW:
    posteriors = posteriors[posteriors['Name'].isin(names)].reset_index(drop=True)

In [ ]:
match_table = build_match_table(hdrs, threshold=THRESHOLD, criterion=CRITERION)
match_table = assign_one_to_one(match_table, method=ASSIGNMENT, cost='6D HDR',
                                posterior_snr=posteriors.set_index('Name')['SNR'] if HAVE_POSTERIOR_SNR else None)
match_table.to_feather(f'{DATA_DIR}/matches_{tag}.feather')
print(f"{match_table['matched'].sum()} matched pairs, {match_table['assigned'].sum()} assigned one-to-one")

In [ ]:
injections = pd.read_feather(f'{DATA_DIR}/injection_set/injections.feather')
if PREVIEW:
    # only injections that are candidates of posteriors with HDRs so far;
    # otherwise injections whose posteriors are still running would show up as "not matched"
    injections = injections[injections['Name'].isin(hdrs['Candidate'])]
inj_summary = injection_summary(match_table, injections)
post_summary = posterior_summary(match_table, posteriors)
inj_summary.to_feather(f'{DATA_DIR}/injection_summary_{tag}.feather')
post_summary.to_feather(f'{DATA_DIR}/posterior_summary_{tag}.feather')
inj_summary['status'].value_counts()

In [ ]:
# 2D histograms of injections by match status: frequency vs log10(SNR) and vs log10(Amplitude)
os.makedirs(f'{PLOTS_DIR}', exist_ok=True)
for y in ['log10(SNR)', 'log10(Amplitude)']:
    fig, hist2d = plot_match_hist2d(inj_summary, x='Frequency (mHz)', y=y, bins=(60, 60), xlim=(0, 15))
    fig.savefig(f"{PLOTS_DIR}/matches_freq_vs_{y.split('(')[1].rstrip(')').lower()}_{tag}.pdf", bbox_inches='tight')
# one panel with all injections instead: plot_match_hist2d(inj_summary, statuses=[None])

## COmpleteness and PUrity 
- **Completeness**: fraction of injections that were assigned to a recovered source, binned in the *injected* frequency and SNR.
- **Purity**: fraction of recovered sources (posteriors) that were assigned an injection, binned in the *recovered* median frequency and catalog SNR.

In [ ]:
post_summary['assigned'] = post_summary['assigned_injection'].notna()
inj_summary['assigned'] = inj_summary['assigned_posterior'].notna()
print(f'purity       = {purity(post_summary):.3f}  ({post_summary.assigned.sum()} / {len(post_summary)} recovered sources)')
print(f'completeness = {completeness(inj_summary):.3f}  ({inj_summary.assigned.sum()} / {len(inj_summary)} injections)')

In [ ]:
hist = {'Catalog': post_summary['Frequency'] * 1e3, 'Injections': inj_summary['Frequency'] * 1e3}

# purity vs recovered frequency and catalog SNR (log10 of median amplitude if no catalog SNR yet)
purity_y = 'log10(SNR)' if HAVE_POSTERIOR_SNR else 'log10(Amplitude)'
fig, purity_map = plot_fraction_map(post_summary, flag='assigned', label='Purity',
                                    y=purity_y, bins=(60, 40), xlim=(0, 20), hist=hist)
fig.axes[0].set_ylabel(f'{purity_y} (catalog, median)')
fig.savefig(f'{PLOTS_DIR}/purity_{tag}.pdf', bbox_inches='tight')

# completeness vs injected frequency and SNR
fig, completeness_map = plot_fraction_map(inj_summary, flag='assigned', label='Completeness',
                                          y='log10(SNR)', bins=(60, 40), xlim=(0, 20), hist=hist)
fig.axes[0].set_ylabel('log10(SNR) (injection)')
fig.savefig(f'{PLOTS_DIR}/completeness_{tag}.pdf', bbox_inches='tight')